# Dual-NSP Audio Streaming

Runs `graphs/dual_nsp_audio.yaml`, which starts two `cerebusAdapter` nodes
(one per NSP, legacy 8-byte packet header -- this rig's NSPs are older
Blackrock hardware, not Gemini-generation). Sample Group assignments come
from `BLACKROCK CONFIG FILES/YFY_spikeAutoThresh.ccf` (parsed with
`parse_ccf.py`) -- a reference config, not a live query. Streams produced:

- `nsp1_neural`: NSP1's neural front-end, Sample Group 3 (2kHz).
- `nsp1_analog`: NSP1's analog-input/BNC bank (includes audio/photodiode
  etc.), Sample Group 5 (30kHz) -- a **separate** Sample Group from
  `nsp1_neural`, so this stream is cleanly just the analog-input bank.
- `nsp2_neural_analog`: NSP2's neural front-end **and** analog-input bank
  are on the *same* Sample Group (5) here, so this one stream is a combined
  272-channel buffer. **Which column is which specific channel (electrode
  vs. BNC/audio) is not yet known** -- the CCF records group membership but
  not intra-packet channel order. Don't index into this stream expecting a
  specific channel until that's resolved (planned via a live cbsdk query,
  in progress separately).

When you stop the graph, all data written to Redis during the run is saved
to a `.pkl` file on disk, following the same pattern as
`brand-tutorial/notebooks/01_calibration.ipynb`.

**Before running:** confirm the above against the *current* live-exported
`.ccf`, not just YFY's -- rerun `parse_ccf.py` on it if config may have
changed. `chan_per_stream` in the graph YAML deliberately uses hardware
ceilings (256/16/272) rather than YFY's exact observed active-channel
counts, so it stays memory-safe even if the live config differs somewhat
(see the memory-safety gotcha in `CLAUDE.md` -- undersizing this corrupts
memory, not just misreads data).

**Buffer layout note:** the legacy `cerebusAdapter` node (unlike
`cerebusAdapterGem`) packs each `samples` entry **sample-major**:
`[samp_per_stream, chan_per_stream]`. The decode cells below use this
orientation -- flip it back if you ever switch to `cerebusAdapterGem`.

**Process launch note:** `supervisor.py`'s own `start_graph()` directly
launches any node whose `machine:` matches its own `-m`, or that has no
`machine:` field at all (like this graph's nodes) -- no separate `booter`
process is needed for this single-machine setup. Just make sure
`supervisor.py` itself is running before the next cell.

## 1. Load the graph

In [1]:
import json
import os
import pickle
import time
from datetime import datetime

import redis
import yaml

DURATION = None  # seconds; leave as None to stop manually with ENTER
GRAPH = '/home/eric/BRAND/brand-modules/brand-bcm-emu/graphs/dual_nsp_audio.yaml'
REDIS_IP = '127.0.0.1'
REDIS_PORT = 6379
notebook_dir = os.getcwd()

with open(GRAPH, 'r') as f:
    graph = yaml.safe_load(f)

graph

{'participant_id': 'bcm_emu',
 'graph_name': 'dual_nsp_audio',
 'session_description': "Stream continuous data (neural + behavioral/analog-input) from both NSPs.\nSample Group assignments below are taken from `BLACKROCK CONFIG FILES/YFY_spikeAutoThresh.ccf` (parsed with `notebooks/parse_ccf.py`) -- this is a REFERENCE config, not a live query of whatever's actually loaded on Central right now. Re-run `parse_ccf.py` against the current live-exported .ccf and update this file if the config has since changed. Long-term plan is to replace this with a live cbsdk query (in progress separately); this file is the interim, static-config-driven version.\n",
 'nodes': [{'name': 'cerebusAdapter',
   'nickname': 'nsp1_in',
   'module': '../brand-modules/brand-nsp',
   'run_priority': 99,
   'cpu_affinity': 2,
   'redis_outputs': ['nsp1_neural', 'nsp1_analog'],
   'parameters': {'broadcast_ip': '192.168.137.255',
    'broadcast_port': 51002,
    'broadcast_device': 'enp4s0',
    'num_streams': 2,
  

## 2. Run the graph

Requires a `supervisor` process already running (see `CLAUDE.md` in the
BRAND repo root -- `supervisor.py` launches nodes with no `machine:` field,
like this graph's, directly itself; no separate `booter` process is needed
for this single-machine setup). This cell starts the graph, waits for you
to hit ENTER, stops the graph, then saves every Redis stream that was
written to during the run into a `.pkl` file under `./data/`.

In [2]:
r = redis.Redis(host=REDIS_IP, port=REDIS_PORT)

curs, start_streams = r.scan(0, _type='stream')
while curs != 0:
    curs, streams = r.scan(curs, _type='stream')
    start_streams += streams

# get the most recent ID from each stream that already exists
start_id = {}
for stream in start_streams:
    replies = r.xrevrange(stream, count=1)
    if replies:
        start_id[stream] = replies[0][0]

print(f'Starting graph from {GRAPH}')
r.xadd('supervisor_ipstream', {
    'commands': 'startGraph',
    'graph': json.dumps(graph)
})

if DURATION:
    print(f'Waiting {DURATION} seconds')
    time.sleep(DURATION)
else:
    input('Hit ENTER to stop graph...')

# Stop the graph
print('Stopping graph')
r.xadd('supervisor_ipstream', {'commands': 'stopGraph'})

curs, stop_streams = r.scan(0, _type='stream')
while curs != 0:
    curs, streams = r.scan(curs, _type='stream')
    stop_streams += streams

new_streams = [
    stream for stream in stop_streams if stream not in start_streams
]
for stream in new_streams:
    start_id[stream] = 0

# Save every stream, from wherever it was before startGraph onward
all_data = {}
for stream in stop_streams:
    all_data[stream] = r.xrange(stream, min=start_id[stream])

date_str = datetime.now().strftime(r'%y%m%dT%H%M')
graph_name = os.path.splitext(os.path.basename(GRAPH))[0]
data_dir = os.path.join(notebook_dir, 'data')
os.makedirs(data_dir, exist_ok=True)
save_path = os.path.join(data_dir, f'{date_str}_{graph_name}.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(all_data, f)
print(f'Saved to {save_path}')
print(f'Saved streams: {sorted(list(all_data.keys()))}')

# Remove the streams that were created during this run so Redis doesn't
# keep accumulating data across repeated runs of this cell
if new_streams:
    while max([r.xlen(stream) for stream in new_streams]):
        for stream in new_streams:
            r.delete(stream)
r.memory_purge()
print(f'Deleted streams: {new_streams}')

Starting graph from /home/eric/BRAND/brand-modules/brand-bcm-emu/graphs/dual_nsp_audio.yaml
Stopping graph
Saved to /home/eric/BRAND/brand-modules/brand-bcm-emu/notebooks/data/260721T1449_dual_nsp_audio.pkl
Saved streams: [b'booter', b'graph_status', b'nsp1_in_state', b'nsp1_nq\x03', b'nsp2_in_state', b'nsp2_neural_analog', b'supergraph_stream', b'supervisor_ipstream', b'supervisor_status', b'\xac\x01\xe1\xe1p']
Deleted streams: [b'nsp1_in_state', b'\xac\x01\xe1\xe1p', b'nsp2_neural_analog', b'nsp2_in_state', b'booter', b'supergraph_stream', b'nsp1_nq\x03', b'supervisor_ipstream']


## 3. Quick sanity check

Loads the `.pkl` back and confirms data actually arrived on both NSPs'
analog streams before you go any further (e.g. writing a full decode
pipeline like `01_calibration.ipynb`'s "Analyze the block" section).

In [3]:
import numpy as np

with open(save_path, 'rb') as f:
    saved_data = pickle.load(f)

# build a stream_name -> per-stream params lookup, since a single node can
# now emit more than one stream (num_streams > 1) with different
# chan_per_stream/samp_per_stream per stream index
stream_params = {}
for node in graph['nodes']:
    p = node['parameters']
    for i, name in enumerate(p['stream_names']):
        stream_params[name] = {
            'chan_per_stream': p['chan_per_stream'][i],
            'samp_per_stream': p['samp_per_stream'][i],
            'samp_freq': p['samp_freq'][i],
        }

for stream_name in [b'nsp1_neural', b'nsp1_analog', b'nsp2_neural_analog']:
    entries = saved_data.get(stream_name, [])
    print(f'{stream_name.decode()}: {len(entries)} entries')
    if not entries:
        continue
    params = stream_params[stream_name.decode()]
    chan_per_stream = params['chan_per_stream']
    samp_per_stream = params['samp_per_stream']

    _, first_entry = entries[0]
    samples = np.frombuffer(first_entry[b'samples'], dtype=np.int16)
    # legacy `cerebusAdapter` (8-byte packet header) does one contiguous
    # memcpy per Cerebus packet -> sample-major layout
    # [samp_per_stream, chan_per_stream]. This is the OPPOSITE of
    # cerebusAdapterGem's channel-major layout -- don't reuse this reshape
    # if you ever switch to the Gem node.
    samples = samples.reshape(samp_per_stream, chan_per_stream)
    print(f'  samples shape (samples_per_packet, channels): {samples.shape}')
    print(f'  channel 0 first few samples: {samples[:5, 0]}')

nsp1_neural: 0 entries
nsp1_analog: 0 entries
nsp2_neural_analog: 242 entries
  samples shape (samples_per_packet, channels): (30, 272)
  channel 0 first few samples: [6404    0    0    0    0]


In [8]:
samples

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

## 4. Pull out an analog-input channel and save as WAV

Extracts a channel from `nsp1_analog` only -- that's the one stream where
the analog-input/BNC bank is cleanly isolated (its own Sample Group,
separate from neural). `nsp2_neural_analog` is deliberately skipped here:
neural and analog-input share one Sample Group on NSP2, so a given column
index isn't known to be the audio channel there yet (see the intro cell).

`AUDIO_CHANNEL_1INDEXED` is still a guess pending confirmation -- it was
never derived from YFY's CCF (that reference session used a photodiode on
this bank, not necessarily the same channel your audio line is on).
Concatenates the channel across every saved entry into one continuous time
series, normalizes by abs-max, and writes a `.wav` so you can just listen
to it and confirm whether real audio is being captured at all -- much
faster than eyeballing raw int16 arrays.

If `packet_type`/`chan_per_stream` are still wrong, this will either error
out (shape mismatch) or produce a suspiciously perfect silent `.wav` --
either is itself useful diagnostic signal, not just a dead end.

In [4]:
import wave

AUDIO_CHANNEL_1INDEXED = 14  # guess, within the 16-ch analog-input bank -- unconfirmed
AUDIO_CHANNEL_IDX = AUDIO_CHANNEL_1INDEXED - 1  # -> 0-indexed column in `samples`

wav_dir = os.path.join(notebook_dir, 'data')
os.makedirs(wav_dir, exist_ok=True)

# nsp2_neural_analog intentionally excluded -- see the markdown cell above
for stream_name in [b'nsp1_analog']:
    entries = saved_data.get(stream_name, [])
    if not entries:
        print(f'{stream_name.decode()}: no entries, skipping')
        continue

    params = stream_params[stream_name.decode()]
    chan_per_stream = params['chan_per_stream']
    samp_per_stream = params['samp_per_stream']
    samp_freq = params['samp_freq']

    # pull out just the audio channel from every entry, in time order
    # (xrange already returns entries oldest-first)
    channel_chunks = []
    for _, entry in entries:
        samples = np.frombuffer(entry[b'samples'], dtype=np.int16)
        # sample-major layout -- see the note in the sanity-check cell above
        samples = samples.reshape(samp_per_stream, chan_per_stream)
        channel_chunks.append(samples[:, AUDIO_CHANNEL_IDX])
    audio = np.concatenate(channel_chunks).astype(np.float32)

    # normalize by abs-max so the loudest sample hits full scale
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio_norm = audio / peak
    else:
        audio_norm = audio
        print(f'{stream_name.decode()}: WARNING -- channel '
              f'{AUDIO_CHANNEL_1INDEXED} (1-indexed) is completely silent '
              f'(all zeros), nothing to normalize')

    pcm16 = (audio_norm * 32767).astype(np.int16)

    wav_path = os.path.join(
        wav_dir, f'{stream_name.decode()}_ch{AUDIO_CHANNEL_1INDEXED}_audio.wav')
    with wave.open(wav_path, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)  # int16
        wf.setframerate(samp_freq)
        wf.writeframes(pcm16.tobytes())

    duration_s = len(pcm16) / samp_freq
    print(f'{stream_name.decode()}: wrote {len(pcm16)} samples '
          f'({duration_s:.2f}s @ {samp_freq} Hz) -> {wav_path}')

nsp1_analog: no entries, skipping
